# Code Complexity vs Agent Resolution Rate

**Hypothesis**: Code that is hard for humans to maintain (high cyclomatic/cognitive complexity, low maintainability index) is also hard for coding agents to fix.

We compute code complexity metrics (via `rust-code-analysis`) on the files touched by each gold patch at the base commit, then correlate with agent resolution rates across multiple runs.

**Metrics examined**:
- Existing baselines: LOC changed in patch, number of files changed
- Cyclomatic complexity, Cognitive complexity
- Halstead volume / effort / estimated bugs
- Maintainability Index (Original, SEI, Visual Studio)
- ABC magnitude (Assignments, Branches, Conditions)
- Source lines of code (SLOC), number of functions

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

sys.path.insert(0, str(Path.cwd().parent))

from utils import DATASET_PATH, count_files_in_patch, count_loc_in_patch, load_aggregate_results

from bcbench.dataset import DatasetEntry, load_dataset_entries

METRICS_PATH = DATASET_PATH.parent / "bcbench_metrics.jsonl"
ANALYSIS_SETUP = "copilot-opus-4-6"

## 1. Load and Merge Data

In [ ]:
# --- Resolution rates from aggregate results (multiple runs) ---
all_results = load_aggregate_results(category="bug-fix")
runs_df = all_results[ANALYSIS_SETUP]

resolution_df = (
    runs_df.groupby("instance_id")["resolved"]
    .agg(resolution_rate="mean", n_runs="count", n_resolved="sum")
    .reset_index()
)

print(f"Setup: {ANALYSIS_SETUP}")
print(f"Instances: {len(resolution_df)}, Runs per instance: {resolution_df['n_runs'].iloc[0]}")
print(f"Overall resolution rate: {resolution_df['resolution_rate'].mean():.1%}")

In [ ]:
# --- Dataset metadata (patch LOC, file count) ---
dataset_entries = load_dataset_entries(DATASET_PATH)

dataset_df = pd.DataFrame(
    [
        {
            "instance_id": e.instance_id,
            "area": e.metadata.area or "Unknown",
            "project": e.extract_project_name(),
            "patch_loc": count_loc_in_patch(e.patch),
            "patch_files": count_files_in_patch(e.patch),
        }
        for e in dataset_entries
    ]
)

print(f"Dataset entries: {len(dataset_df)}")

In [ ]:
# --- Code complexity metrics (from rust-code-analysis) ---
metrics_rows = []
for line in METRICS_PATH.read_text(encoding="utf-8").splitlines():
    entry = json.loads(line)
    files_with_metrics = [f for f in entry["files"] if f["metrics"] is not None]

    if not files_with_metrics:
        # All files are new — use zeros
        metrics_rows.append({"instance_id": entry["instance_id"]})
        continue

    # Aggregate across files: sum for additive metrics, weighted avg for rates
    row = {"instance_id": entry["instance_id"]}
    all_m = [f["metrics"] for f in files_with_metrics]

    # Summed metrics (additive across files)
    row["cyclomatic_sum"] = sum(m["cyclomatic"]["sum"] for m in all_m)
    row["cyclomatic_max"] = max(m["cyclomatic"]["max"] for m in all_m)
    row["cognitive_sum"] = sum(m["cognitive"]["sum"] for m in all_m)
    row["cognitive_max"] = max(m["cognitive"]["max"] for m in all_m)
    row["halstead_volume"] = sum(m["halstead"]["volume"] for m in all_m)
    row["halstead_effort"] = sum(m["halstead"]["effort"] for m in all_m)
    row["halstead_difficulty"] = max(m["halstead"]["difficulty"] for m in all_m)
    row["halstead_bugs"] = sum(m["halstead"]["bugs"] for m in all_m)
    row["sloc"] = sum(m["loc"]["sloc"] for m in all_m)
    row["ploc"] = sum(m["loc"]["ploc"] for m in all_m)
    row["num_functions"] = sum(m["nom"]["functions"] for m in all_m)
    row["abc_magnitude"] = sum(m["abc"]["magnitude"] for m in all_m)
    row["nexits_sum"] = sum(m["nexits"]["sum"] for m in all_m)
    row["nargs_total"] = sum(m["nargs"]["total"] for m in all_m)

    # Averaged metrics (normalized per-file, take weighted mean by SLOC)
    total_sloc = row["sloc"] or 1
    for mi_key in ["mi_original", "mi_sei", "mi_visual_studio"]:
        row[mi_key] = sum(
            m["mi"][mi_key] * m["loc"]["sloc"] for m in all_m
        ) / total_sloc

    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows).fillna(0)
print(f"Metrics entries: {len(metrics_df)}")
print(f"Columns: {list(metrics_df.columns)}")

In [ ]:
# --- Merge everything ---
df = resolution_df.merge(dataset_df, on="instance_id").merge(metrics_df, on="instance_id")
print(f"Merged dataset: {len(df)} instances")
df.head(3)

## 2. Correlation Analysis

We use **Spearman rank correlation** between each metric and the per-instance resolution rate. Spearman is appropriate here because:
- We don't assume a linear relationship
- It's robust to outliers (some files have extreme SLOC or complexity)
- Resolution rate is bounded [0, 1] so rank-based correlation is more reliable

In [ ]:
METRIC_COLUMNS = [
    # Existing baselines
    "patch_loc",
    "patch_files",
    # Complexity metrics
    "cyclomatic_sum",
    "cyclomatic_max",
    "cognitive_sum",
    "cognitive_max",
    # Halstead
    "halstead_volume",
    "halstead_effort",
    "halstead_difficulty",
    "halstead_bugs",
    # Scale
    "sloc",
    "num_functions",
    # Maintainability (higher = more maintainable, expect positive correlation)
    "mi_original",
    "mi_sei",
    "mi_visual_studio",
    # Other
    "abc_magnitude",
    "nexits_sum",
    "nargs_total",
]

METRIC_LABELS = {
    "patch_loc": "Patch LOC (baseline)",
    "patch_files": "Files Changed (baseline)",
    "cyclomatic_sum": "Cyclomatic Complexity (sum)",
    "cyclomatic_max": "Cyclomatic Complexity (max fn)",
    "cognitive_sum": "Cognitive Complexity (sum)",
    "cognitive_max": "Cognitive Complexity (max fn)",
    "halstead_volume": "Halstead Volume",
    "halstead_effort": "Halstead Effort",
    "halstead_difficulty": "Halstead Difficulty",
    "halstead_bugs": "Halstead Est. Bugs",
    "sloc": "Source Lines of Code",
    "num_functions": "Number of Functions",
    "mi_original": "Maintainability Index (Original)",
    "mi_sei": "Maintainability Index (SEI)",
    "mi_visual_studio": "Maintainability Index (VS)",
    "abc_magnitude": "ABC Magnitude",
    "nexits_sum": "Exit Points (sum)",
    "nargs_total": "Function Arguments (total)",
}

correlations = []
for col in METRIC_COLUMNS:
    rho, p_value = stats.spearmanr(df[col], df["resolution_rate"])
    correlations.append({
        "metric": METRIC_LABELS.get(col, col),
        "column": col,
        "spearman_rho": round(rho, 3),
        "p_value": round(p_value, 4),
        "significant": p_value < 0.05,
        "direction": "higher complexity → lower resolution" if rho < 0 else "higher metric → higher resolution",
    })

corr_df = pd.DataFrame(correlations).sort_values("p_value")
corr_df[["metric", "spearman_rho", "p_value", "significant", "direction"]]

In [ ]:
# Visualize correlation coefficients
plot_df = corr_df.sort_values("spearman_rho")

fig = go.Figure()
fig.add_trace(go.Bar(
    y=plot_df["metric"],
    x=plot_df["spearman_rho"],
    orientation="h",
    marker_color=[
        "#2ecc71" if sig and rho > 0
        else "#e74c3c" if sig and rho < 0
        else "#bdc3c7"
        for sig, rho in zip(plot_df["significant"], plot_df["spearman_rho"])
    ],
    text=[f"ρ={r:.3f} (p={p:.3f})" for r, p in zip(plot_df["spearman_rho"], plot_df["p_value"])],
    textposition="outside",
))

fig.update_layout(
    title=f"Spearman Correlation: Code Metrics vs Resolution Rate ({ANALYSIS_SETUP})",
    xaxis_title="Spearman ρ (negative = harder code → lower resolution)",
    yaxis_title="",
    height=600,
    width=900,
    margin=dict(l=250),
    xaxis=dict(range=[-0.5, 0.5], zeroline=True, zerolinecolor="black", zerolinewidth=1),
)
fig.show()

## 3. Binned Resolution Rates

For the most promising metrics, bin instances into groups and show resolution rates per bin. This makes the relationship easier to interpret than raw correlation numbers.

In [ ]:
def resolution_by_bins(df: pd.DataFrame, col: str, bins: list, labels: list, col_label: str) -> pd.DataFrame:
    """Bin a metric column and compute resolution rate per bin."""
    binned = pd.cut(df[col], bins=bins, labels=labels)
    result = (
        df.assign(bin=binned)
        .groupby("bin", observed=True)
        .agg(
            resolution_rate=("resolution_rate", "mean"),
            count=("instance_id", "count"),
        )
        .reset_index()
    )
    total = result["count"].sum()
    result["% Resolved"] = (result["resolution_rate"] * 100).round(1).astype(str) + "%"
    result["Instances"] = result["count"].astype(str) + " (" + (result["count"] / total * 100).round(1).astype(str) + "%)"
    result = result.rename(columns={"bin": col_label})
    return result


# --- Patch LOC (baseline) ---
loc_bins = resolution_by_bins(df, "patch_loc", [0, 5, 10, 50, float("inf")], ["1-5", "6-10", "11-50", "50+"], "Patch LOC")
print("Resolution Rate by Patch LOC (baseline):")
print(loc_bins[["Patch LOC", "% Resolved", "Instances"]].to_string(index=False))

print()

# --- Files changed (baseline) ---
files_bins = resolution_by_bins(df, "patch_files", [0, 1, 2, float("inf")], ["1", "2", "3+"], "Files Changed")
print("Resolution Rate by Files Changed (baseline):")
print(files_bins[["Files Changed", "% Resolved", "Instances"]].to_string(index=False))

In [ ]:
# --- Cyclomatic Complexity ---
cyc_bins = resolution_by_bins(
    df, "cyclomatic_sum",
    [0, 50, 150, 500, float("inf")],
    ["1-50", "51-150", "151-500", "500+"],
    "Cyclomatic Complexity"
)
print("Resolution Rate by Cyclomatic Complexity (of touched files):")
print(cyc_bins[["Cyclomatic Complexity", "% Resolved", "Instances"]].to_string(index=False))

print()

# --- Cognitive Complexity ---
cog_bins = resolution_by_bins(
    df, "cognitive_sum",
    [0, 25, 100, 300, float("inf")],
    ["1-25", "26-100", "101-300", "300+"],
    "Cognitive Complexity"
)
print("Resolution Rate by Cognitive Complexity (of touched files):")
print(cog_bins[["Cognitive Complexity", "% Resolved", "Instances"]].to_string(index=False))

In [ ]:
# --- Maintainability Index ---
mi_bins = resolution_by_bins(
    df, "mi_visual_studio",
    [-float("inf"), 10, 20, 40, float("inf")],
    ["<10 (very low)", "10-20 (low)", "20-40 (moderate)", "40+ (high)"],
    "Maintainability Index (VS)"
)
print("Resolution Rate by Maintainability Index (VS):")
print(mi_bins[["Maintainability Index (VS)", "% Resolved", "Instances"]].to_string(index=False))

print()

# --- SLOC ---
sloc_bins = resolution_by_bins(
    df, "sloc",
    [0, 200, 500, 2000, float("inf")],
    ["1-200", "201-500", "501-2000", "2000+"],
    "SLOC (touched files)"
)
print("Resolution Rate by SLOC (of touched files):")
print(sloc_bins[["SLOC (touched files)", "% Resolved", "Instances"]].to_string(index=False))

In [ ]:
# --- Halstead Estimated Bugs ---
hb_bins = resolution_by_bins(
    df, "halstead_bugs",
    [0, 0.5, 2, 10, float("inf")],
    ["<0.5", "0.5-2", "2-10", "10+"],
    "Halstead Est. Bugs"
)
print("Resolution Rate by Halstead Estimated Bugs:")
print(hb_bins[["Halstead Est. Bugs", "% Resolved", "Instances"]].to_string(index=False))

print()

# --- Number of Functions ---
fn_bins = resolution_by_bins(
    df, "num_functions",
    [0, 10, 30, 100, float("inf")],
    ["1-10", "11-30", "31-100", "100+"],
    "Num Functions"
)
print("Resolution Rate by Number of Functions (in touched files):")
print(fn_bins[["Num Functions", "% Resolved", "Instances"]].to_string(index=False))

## 4. Scatter Plots — Top Metrics vs Resolution Rate

In [ ]:
# Pick the top 4 most correlated metrics (by absolute rho)
top_metrics = corr_df.reindex(corr_df["spearman_rho"].abs().sort_values(ascending=False).index).head(4)

for _, row in top_metrics.iterrows():
    col = row["column"]
    label = row["metric"]
    rho = row["spearman_rho"]
    p = row["p_value"]

    fig = px.scatter(
        df, x=col, y="resolution_rate",
        hover_data=["instance_id", "project", "area"],
        labels={col: label, "resolution_rate": "Resolution Rate"},
        title=f"{label} vs Resolution Rate (ρ={rho:.3f}, p={p:.4f})",
        trendline="lowess",
        trendline_color_override="red",
    )
    fig.update_layout(height=400, width=700, yaxis=dict(range=[-0.05, 1.05]))
    fig.show()

## 5. Cross-Setup Comparison

Do the correlations hold across different agent setups? If so, the relationship between code complexity and agent performance is robust.

In [ ]:
all_agg = load_aggregate_results(category="bug-fix")

cross_setup_rows = []
for setup_name, setup_df in all_agg.items():
    setup_resolution = (
        setup_df.groupby("instance_id")["resolved"]
        .mean()
        .reset_index()
        .rename(columns={"resolved": "resolution_rate"})
    )
    merged = setup_resolution.merge(metrics_df, on="instance_id")
    merged = merged.merge(dataset_df[["instance_id", "patch_loc", "patch_files"]], on="instance_id")

    for col in METRIC_COLUMNS:
        rho, p_value = stats.spearmanr(merged[col], merged["resolution_rate"])
        cross_setup_rows.append({
            "setup": setup_name,
            "metric": METRIC_LABELS.get(col, col),
            "spearman_rho": round(rho, 3),
            "p_value": round(p_value, 4),
            "significant": p_value < 0.05,
        })

cross_df = pd.DataFrame(cross_setup_rows)

# Pivot: metric × setup → rho
pivot = cross_df.pivot_table(index="metric", columns="setup", values="spearman_rho")

# Sort by average absolute correlation
pivot["avg_abs_rho"] = pivot.abs().mean(axis=1)
pivot = pivot.sort_values("avg_abs_rho", ascending=False)
pivot = pivot.drop(columns="avg_abs_rho")

print("Spearman ρ across setups (sorted by average |ρ|):")
pivot

In [ ]:
# Heatmap of correlations across setups
fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale="RdBu",
    zmid=0,
    zmin=-0.4,
    zmax=0.4,
    text=[[f"{v:.3f}" for v in row] for row in pivot.values],
    texttemplate="%{text}",
    textfont=dict(size=11),
))

fig.update_layout(
    title="Code Metrics vs Resolution Rate — Correlation Across Agent Setups",
    height=650,
    width=800,
    margin=dict(l=250),
    yaxis=dict(autorange="reversed"),
)
fig.show()

## 6. Summary

Key findings printed below — check which complexity metrics outperform the simple baselines (patch LOC, files changed).

In [ ]:
# Compare baselines vs complexity metrics
primary_corr = corr_df.set_index("column")
baseline_rho = max(abs(primary_corr.loc["patch_loc", "spearman_rho"]), abs(primary_corr.loc["patch_files", "spearman_rho"]))

print("=" * 70)
print("SUMMARY: Code Complexity vs Agent Resolution Rate")
print("=" * 70)
print(f"\nBaseline correlations (Spearman ρ with resolution rate):")
print(f"  Patch LOC:     ρ = {primary_corr.loc['patch_loc', 'spearman_rho']:.3f} (p = {primary_corr.loc['patch_loc', 'p_value']:.4f})")
print(f"  Files Changed: ρ = {primary_corr.loc['patch_files', 'spearman_rho']:.3f} (p = {primary_corr.loc['patch_files', 'p_value']:.4f})")
print(f"\nBest baseline |ρ|: {baseline_rho:.3f}")

better = corr_df[corr_df["spearman_rho"].abs() > baseline_rho].sort_values("spearman_rho", key=abs, ascending=False)
if not better.empty:
    print(f"\nMetrics with STRONGER correlation than baselines:")
    for _, r in better.iterrows():
        sig_marker = "*" if r["significant"] else " "
        print(f"  {sig_marker} {r['metric']}: ρ = {r['spearman_rho']:.3f} (p = {r['p_value']:.4f})")
else:
    print("\nNo complexity metrics had stronger correlation than the baselines.")

sig_metrics = corr_df[corr_df["significant"]]
print(f"\nStatistically significant metrics (p < 0.05): {len(sig_metrics)} / {len(corr_df)}")
if not sig_metrics.empty:
    for _, r in sig_metrics.iterrows():
        print(f"  {r['metric']}: ρ = {r['spearman_rho']:.3f} (p = {r['p_value']:.4f}) — {r['direction']}")